In [1]:
!git clone https://github.com/Black-Hawk21/DeepFashion-In-shop-Clothes-Retrieval.git
%cd DeepFashion-In-shop-Clothes-Retrieval

Cloning into 'DeepFashion-In-shop-Clothes-Retrieval'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 90 (delta 30), reused 74 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 61.92 KiB | 2.29 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/kaggle/working/DeepFashion-In-shop-Clothes-Retrieval


In [2]:
!pip install ftfy regex tqdm omegaconf
!pip install git+https://github.com/openai/CLIP.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.6 MB/s eta 0:00:00
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-zpwfjxua
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-zpwfjxua
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=cd34cd188e7b2ac23eee5cc44cf06401afff2a33366f1b66be04552bd4f887b8
  Stored in directory: /tmp/pip-ephem-wheel-cache-o5zsl_rl/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [3]:
import os
print(os.listdir("/kaggle/input"))

['datasets']


In [4]:
from omegaconf import OmegaConf

cfg_path = "configs/config.yaml"
cfg = OmegaConf.load(cfg_path)

base = "/kaggle/input/datasets/shubhranilbasak/deepfashion-inshop/deepfashion"

# FIX: point to parent, NOT img/
cfg.paths.img_dir = base
cfg.paths.partition_file = f"{base}/Eval/list_eval_partition.txt"
cfg.paths.checkpoint_dir = "/kaggle/working/checkpoints"
cfg.paths.results_dir = "/kaggle/working/results"

OmegaConf.save(cfg, cfg_path)

print(cfg.paths)

{'data_root': 'data/deepfashion', 'img_dir': '/kaggle/input/datasets/shubhranilbasak/deepfashion-inshop/deepfashion', 'partition_file': '/kaggle/input/datasets/shubhranilbasak/deepfashion-inshop/deepfashion/Eval/list_eval_partition.txt', 'bbox_file': 'data/deepfashion/Anno/list_bbox_inshop.txt', 'checkpoint_dir': '/kaggle/working/checkpoints', 'index_dir': 'index', 'results_dir': '/kaggle/working/results'}


In [5]:
!mkdir -p /kaggle/working/checkpoints
!mkdir -p /kaggle/working/results

In [6]:
import os

base = "/kaggle/input/datasets/shubhranilbasak/deepfashion-inshop/deepfashion"

print("Base:", os.listdir(base))
print("Img sample:", os.listdir(base + "/img")[:5])
print("Eval:", os.listdir(base + "/Eval"))

Base: ['Eval', 'img', 'Anno']
Img sample: ['MEN', 'WOMEN']
Eval: ['list_eval_partition.txt']


In [7]:
!python scripts/train_clip.py --config configs/config.yaml

[Seed] All seeds set to 510
[Device] Using GPU: Tesla T4
[2026-05-13 05:35:37] [INFO] Config: configs/config.yaml  Seed: 510
100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 169MiB/s]
[CLIP] Unfroze last 4 vision blocks (total=12) + ln_post + proj
[CLIP] Trainable: 28,746,240 / 151,277,313 params (19.00%)
[2026-05-13 05:35:44] [INFO] Train/val split: train_samples=23180  val_samples=2702  val_query=399  val_gallery=2303  val_items_skipped=0
/kaggle/working/DeepFashion-In-shop-Clothes-Retrieval/scripts/train_clip.py:282: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=cfg.train.use_amp)
[DeepFashionDataset] split=train  samples=23180  unique_items=3598
[2026-05-13 05:35:44] [INFO] Starting training for 20 epochs with loss=infonce, α_temp=0.07
Epoch 1:   0%|                                          | 0/362 [00:00<?, ?it/s]/kaggle/working/DeepFashion-In-shop-C